# Erkennung von Hülsenfrüchten in Bildern

In diesem Projekt geht es um die Erkennung von Hülsenfrüchten in von den Studierenden selbst aufgenommenen Bildern. Ziel ist es die Hülsenfrüchte im Bild zu identifizieren und gemäß Ihrer Klasse unterschiedlich zu markieren.


### Hinweis

Bewusst wurde dise Jupyther Notebook Anleitung sehr offen gehalten, um den Studierendern die Möglichkeit zu geben, eigene Ideen und Ansätze einzubringen. Wenn möglich wurde auf feste Umsetzungsvorgaben verzichtet.

#### Package Import

In [1]:
# package import
%matplotlib ipympl
import numpy as np
import matplotlib.pyplot as plt
import cv2
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import skimage
import matplotlib.pyplot as plt
import matplotlib

#### Einladen und Anzeige des Bildes

In [2]:
img = skimage.io.imread(r'img/beans/AllBeans1.jpg', False)

dataType = img.dtype
imageShape = img.shape
print(f'Datentyp des Bildes: {dataType}')
print(f'fortmat des Bildes: {imageShape}')

fig = px.imshow(img)
fig.show()

Datentyp des Bildes: uint8
fortmat des Bildes: (3000, 4000, 3)


#### Bildvorverarbeitung

In [3]:
def imagePreprocessing(img):
    img_blur = cv2.blur(img,(25,25))
    
    return img_blur

**Anzeige des Ergebnisses**

In [4]:
image_Preprocessed = imagePreprocessing(img)
fig = px.imshow(image_Preprocessed)
fig.show()

#### Bildhauptverarbeitung

In [5]:
### Definieren Sie Ihre Hauptverarbeitungsfunktionen hier.

def convertToHSV(img):
    img_hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    return img_hsv


def seperateGreenBkg(img_hsv, low, high):

    mask_hue = 1-((img_hsv[:,:,0]>low[0])&(img_hsv[:,:,0]<high[0]))
    
    mask_sat = 1-((img_hsv[:,:,1]>low[1])&(img_hsv[:,:,1]<high[1]))
    
    mask_val = ((img_hsv[:,:,2]>low[2])&(img_hsv[:,:,2]<high[2]))
    
    mask =(mask_hue & mask_sat & mask_val)
    mask = mask.astype('uint8')

    '''
    mask = cv2.inRange(img_hsv, low, high, )
    der obige code funktioniert nicht mit einem einzigen cv2.inRange befehl.
    '''
    return mask

def cleanMask(mask):
    # clean region
    footprint = skimage.morphology.disk(15)
    mask_clean = cv2.morphologyEx(mask, cv2.MORPH_OPEN, footprint)
    mask_clean = cv2.morphologyEx(mask_clean, cv2.MORPH_CLOSE, footprint)

    return mask_clean

def imageSegmentation(mask):
    
    #segments = watershedSegmentation(mask)
    segments = skimage.measure.label(mask)
    #segments = cv2.connectedComponents(mask)[1]
    return segments

    return segmented_image


def mainImageProcessing(img):
    # Fügen Sie hier Ihre Hauptverarbeitung hinzu

    img_hsv = convertToHSV(img)


    low = np.array([20, 200, 40])
    high = np.array([105, 255, 255])
    mask = seperateGreenBkg(img_hsv, low, high)
    mask = cleanMask(mask)
    img_segmented = imageSegmentation(mask)

    return img_segmented

In [6]:
image_processed = mainImageProcessing(image_Preprocessed)

fig = px.imshow(image_processed)
fig.show()

##### Markieren der Detektierten Objekte im Bild
*Bestimmung der Position*

In [7]:
def calculateAllRegionCenters(imageLabeled):

    # iterate over all regions
    noOfRegions = np.max(imageLabeled)+1
    centerOfRegions = np.zeros((noOfRegions-1, 2))
    for regionId in range(1, noOfRegions):
        centroid = skimage.measure.centroid(imageLabeled == regionId)
        centerOfRegions[regionId-1,:] = centroid
    return centerOfRegions

In [8]:


centersOfRegions = calculateAllRegionCenters(image_processed)

fig = px.imshow(img)
fig.add_trace(
    go.Scatter(
        x=centersOfRegions[:,1],
        y=centersOfRegions[:,0],
        mode='markers+text',
        marker=dict(size=14, color='red', symbol='cross'),
        textposition='middle center',
        name='Hülsenfrucht',
        showlegend = True
    )
)
fig.show()

#### Analyse der Bildinformationen

##### Aufbau eines Feature Vektors

In [9]:
## Feature Vektor für die gefundenen Objekte erstellen. Numpy array so aufbauen dass die Zeilen den Objekten entsprechen und die Spalten den Features.
## Als Feature verwenden wie die Farbe / helligkeit der Früchte und die Elliptizität.

def getElipticity(imageLabeled):
    
    noOfRegions = np.max(imageLabeled)+1
    elipticityOfRegion = np.zeros(noOfRegions-1)
    regions = skimage.measure.regionprops(imageLabeled)
    for regionId in range(1, noOfRegions):
        reg = regions[regionId-1]
        major_axis = reg.major_axis_length
        minor_axis = reg.minor_axis_length
        ellipticity = 1.0 - (minor_axis / major_axis) if major_axis > 0 else 0.0
        elipticityOfRegion[regionId-1] = ellipticity
    return elipticityOfRegion

def getAverageIntensityValue(image, mask):
    noOfRegions = np.max(mask)+1
    regionAV = np.zeros(noOfRegions-1)
    for region in range(1,noOfRegions):
        mean = np.mean(image[mask == region])
        regionAV[region-1] = mean
    return regionAV

def buildFeatureVector(image, imageLabeled):
    ellipticityInRegion = getElipticity(imageLabeled)
    avIntInRegion = getAverageIntensityValue(image, imageLabeled)
    avIntInRegion = avIntInRegion/255
    featureVector = np.hstack((avIntInRegion[:,np.newaxis], ellipticityInRegion[:,np.newaxis]))
    return featureVector

In [ ]:
featureVector = buildFeatureVector(img, image_processed)


# Referenzfeature bauen sorzten sind getrennt zeilenweise

## build Feature Reference

featureRedBean = np.array([0.3, 0.5])
featureWhiteBean = np.array([0.8, 0.5])
featureChickPea = np.array([0.8, 0.08])

featureReference=np.vstack((featureRedBean[np.newaxis,:],
                            featureWhiteBean[np.newaxis,:],
                            featureChickPea[np.newaxis,:]))

print(f'Feature Reference: {featureReference}')

Feature Reference: [[0.3  0.5 ]
 [0.8  0.5 ]
 [0.8  0.08]]


Analyse der Sorte durch Vergleich mit der Referenz

In [13]:
def getSorte(featReference, featureVector):
    row, col = featureVector.shape
    whatKindOfBean = np.zeros((row))*np.nan
    for region in range(row):
        diff = featReference-featureVector[region,:]
        distance = np.linalg.norm(diff, axis=1)
        objectNumber = np.argmin(distance)
        whatKindOfBean[region] = objectNumber
    return whatKindOfBean

In [15]:
sorte = getSorte(featureReference, featureVector)
anzahlRote = np.sum(sorte == 0)
anzahlWeisse = np.sum(sorte == 1)
anzahlKichererbsen = np.sum(sorte == 2)
print(f'Anzahl Rote Bohnen: {anzahlRote}')
print(f'Anzahl Weisse Bohnen: {anzahlWeisse}')
print(f'Anzahl Kichererbsen: {anzahlKichererbsen}')

Anzahl Rote Bohnen: 4
Anzahl Weisse Bohnen: 4
Anzahl Kichererbsen: 11


In [20]:
fig = px.imshow(img)
fig.add_trace(
    go.Scatter(
        x=centersOfRegions[sorte==0,1],
        y=centersOfRegions[sorte==0,0],
        mode='markers+text',
        marker=dict(size=14, color='red', symbol='triangle-up'),
        textposition='middle center',
        name='Rote Bohne',
        showlegend = True
    )
)


fig.add_trace(
    go.Scatter(
        x=centersOfRegions[sorte==1,1],
        y=centersOfRegions[sorte==1,0],
        mode='markers+text',
        marker=dict(size=14, color='blue', symbol='diamond'),
        textposition='middle center',
        name='Weisse Bohne',
        showlegend = True
    )
)

fig.add_trace(
    go.Scatter(
        x=centersOfRegions[sorte==2,1],
        y=centersOfRegions[sorte==2,0],
        mode='markers+text',
        marker=dict(size=14, color='green', symbol='circle'),
        textposition='middle center',
        name='Kichererbsen',
        showlegend = True
    )
)

fig.update_layout(
    width=1200,
    height=800,
    margin=dict(l=0, r=20, t=20, b=20)
)
fig.update_xaxes(constrain="domain", constraintoward="left")
fig.show()

